# AM5061 · Week 12 · Solar absorption chiller

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "glide", "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
    "lmtd", "effectiveness", "ntu_required", "exergy", "T0_REF", "P0_REF",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


# ------------------------------------------------- exchanger relations
def lmtd(dT1, dT2):
    """Log-mean temperature difference.

    Falls back to the arithmetic mean when the two ends are within 1% of each
    other, where the log form is numerically unstable and the two agree to
    better than 0.01% anyway.
    """
    dT1, dT2 = float(dT1), float(dT2)
    if dT1 <= 0 or dT2 <= 0:
        raise ValueError(
            f"temperature difference must be positive at both ends "
            f"(got {dT1:g} and {dT2:g}). A non-positive end means the streams "
            "cross, which no exchanger of this configuration can do."
        )
    if abs(dT1 - dT2) < 0.01*max(dT1, dT2):
        return 0.5*(dT1 + dT2)
    return (dT1 - dT2)/math.log(dT1/dT2)


def effectiveness(config, NTU, Cr):
    """Effectiveness for the standard configurations.

    config: 'counter', 'parallel', 'shell1'  (one shell pass, 2/4/... tube passes),
            'cross-both-unmixed' (approximate), 'cross-Cmax-mixed', 'cross-Cmin-mixed'

    Cr = C_min/C_max. Cr = 0 is the phase-change limit and every configuration
    collapses to the same expression, which is why boilers and condensers are
    easy and everything else is not.
    """
    if NTU < 0:
        raise ValueError("NTU cannot be negative")
    if not 0 <= Cr <= 1:
        raise ValueError(f"Cr must be between 0 and 1, got {Cr:g}")
    if Cr == 0:                       # phase change on one side
        return 1 - math.exp(-NTU)
    if config == "counter":
        if abs(Cr - 1) < 1e-12:
            return NTU/(1 + NTU)
        e = math.exp(-NTU*(1 - Cr))
        return (1 - e)/(1 - Cr*e)
    if config == "parallel":
        return (1 - math.exp(-NTU*(1 + Cr)))/(1 + Cr)
    if config == "shell1":
        r = math.sqrt(1 + Cr*Cr)
        e = math.exp(-NTU*r)
        return 2/(1 + Cr + r*(1 + e)/(1 - e))
    if config == "cross-both-unmixed":
        return 1 - math.exp((math.exp(-Cr*NTU**0.78) - 1)*NTU**0.22/Cr)
    if config == "cross-Cmax-mixed":
        return (1/Cr)*(1 - math.exp(-Cr*(1 - math.exp(-NTU))))
    if config == "cross-Cmin-mixed":
        return 1 - math.exp(-(1 - math.exp(-Cr*NTU))/Cr)
    raise ValueError(f"unknown configuration {config!r}")


def ntu_required(config, eps, Cr, hi=200.0):
    """Invert effectiveness() for NTU. Design direction, rather than rating."""
    eps_max = effectiveness(config, hi, Cr)
    if eps >= eps_max:
        raise ValueError(
            f"effectiveness {eps:g} is unreachable for {config} at Cr={Cr:g}; "
            f"the limit as NTU->infinity is {eps_max:.6f}. "
            "Change the configuration or accept less."
        )
    return solve(lambda n: effectiveness(config, n, Cr) - eps, 1.0,
                 bracket=(1e-9, hi))


# --------------------------------------------------------------- exergy
T0_REF, P0_REF = 303.15, 101325.0      # 30 C, sea level: the Chennai dead state


def exergy(st, T0=T0_REF, p0=P0_REF):
    """Specific flow exergy, J/kg:  (h - h0) - T0*(s - s0).

    The dead state is the ambient the plant actually sits in, so it is a
    DESIGN CHOICE, not a constant. Report which one you used - a Chennai
    dead state and a European one give different answers for the same plant.
    """
    ref = State(st.fluid, T=T0, P=p0)
    return (st.h - ref.h) - T0*(st.s - ref.s)


---
## The case

Grid-independent cooling for **500 tonnes of potato storage in Agra**. Evacuated
tube collectors drive a **single-effect LiBr–water absorption chiller**. Size the
collector field.

Deliverable **D-12**: collector area and storage volume for an **80% solar
fraction**.

### What is different about an absorption chiller

There is no compressor. The pressure lift is done by a **liquid pump** on a
solution loop, which costs almost nothing, and the "compression" is done by
**heat** in the generator. So the machine is driven by a temperature, not by
electricity, and its COP is around **0.7 — not 3**. That is not a bad number;
it is a different quantity, because the input is low-grade heat.

> **Property note.** CoolProp supplies LiBr **solution** properties (density,
> specific heat, enthalpy). It does **not** supply the vapour–liquid
> equilibrium, so the solution concentrations here are **design inputs read off
> the Dühring chart**, which is exactly how a first-pass design is done. Reading
> them off the chart is your job, and the chart is on the Week 12 slides.


## 1. The cold store load

In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
from scipy.optimize import brentq
am.style_plots()

MASS_T      = 500.0        # tonnes of potato
T_store     = am.K(4)      # storage temperature
respiration = 25.0         # W per tonne, potato at 4 C (typical)
transmission= 12.0e3       # W, envelope gain
infiltration= 4.0e3        # W
Q_field     = 18.0e3       # W, field heat pull-down averaged over the season

Q_cool = MASS_T*respiration + transmission + infiltration
print(f"  respiration   {MASS_T*respiration/1e3:7.2f} kW")
print(f"  transmission  {transmission/1e3:7.2f} kW")
print(f"  infiltration  {infiltration/1e3:7.2f} kW")
print(f"  steady load   {Q_cool/1e3:7.2f} kW  = {Q_cool/3516.85:.1f} TR")


## 2. The single-effect cycle

Four temperatures fix the machine: evaporator, condenser, absorber and
generator. The two **concentrations** come off the Dühring chart at those
temperatures — enter them here.


In [ ]:
T_evap  = am.K(5)       # chilled water leaves ~7 C for a 4 C store
T_cond  = am.K(40)      # cooling tower water + approach
T_abs   = am.K(38)
T_gen   = am.K(88)      # driven by evacuated tubes

# ---- FROM THE DUHRING CHART -----------------------------------------
# Read these at (T_abs, T_evap) and (T_gen, T_cond). Typical single-effect
# values for the temperatures above; replace with your own chart readings.
X_weak   = 0.56        # kg LiBr / kg solution, leaving the ABSORBER
X_strong = 0.62        # leaving the GENERATOR
# ---------------------------------------------------------------------

if X_strong <= X_weak:
    raise ValueError("the strong solution must be more concentrated than the weak")
f = X_strong/(X_strong - X_weak)      # circulation ratio, pure LiBr mass balance
print(f"  weak solution   X = {X_weak:.3f}   (from the absorber)")
print(f"  strong solution X = {X_strong:.3f}   (from the generator)")
print(f"  circulation ratio f = X_s/(X_s - X_w) = {f:.3f}")
print(f"\n  {f:.1f} kg of solution must be pumped for every 1 kg of refrigerant")
print("  vapour boiled off. Narrow the concentration gap and f explodes, which")
print("  is why the crystallisation limit matters so much.")


### The crystallisation limit

Push the strong solution too concentrated, or cool it too far, and LiBr
**crystallises** and blocks the machine. That is the hard boundary on the right
of the Dühring chart, and it is why absorption chillers are fussy about cooling
water temperature.


In [ ]:
def solution_h(T, X):
    """Enthalpy of aqueous LiBr from CoolProp's incompressible solution data."""
    return PropsSI("H", "T", T, "P", 1e5, f"INCOMP::LiBr[{X:.4f}]")

def chiller(Q_cool, T_evap=T_evap, T_cond=T_cond, T_gen=T_gen, T_abs=T_abs,
            X_w=X_weak, X_s=X_strong, eta_shx=0.65):
    fcirc = X_s/(X_s - X_w)
    # refrigerant (pure water) side
    h_v_gen  = PropsSI("H","T",T_gen, "Q",1,"Water")      # vapour off the generator
    h_l_cond = PropsSI("H","T",T_cond,"Q",0,"Water")      # condensate
    h_v_evap = PropsSI("H","T",T_evap,"Q",1,"Water")
    m_ref    = Q_cool/(h_v_evap - h_l_cond)               # isenthalpic throttle
    Q_cond   = m_ref*(h_v_gen - h_l_cond)

    # solution side
    m_weak, m_strong = fcirc*m_ref, (fcirc - 1)*m_ref
    h_w_abs = solution_h(T_abs, X_w)
    h_s_gen = solution_h(T_gen, X_s)
    # solution heat exchanger preheats the weak stream using the hot strong one
    dT_max  = T_gen - T_abs
    h_w_in  = h_w_abs + eta_shx*dT_max*PropsSI("C","T",T_abs,"P",1e5,f"INCOMP::LiBr[{X_w:.4f}]")
    h_s_out = h_s_gen - (m_weak/max(m_strong,1e-9))*(h_w_in - h_w_abs)

    Q_gen = m_ref*h_v_gen + m_strong*h_s_gen - m_weak*h_w_in
    Q_abs = m_ref*h_v_evap + m_strong*h_s_out - m_weak*h_w_abs
    return {"Q_cool, kW": Q_cool/1e3, "Q_gen, kW": Q_gen/1e3,
            "Q_cond, kW": Q_cond/1e3, "Q_abs, kW": Q_abs/1e3,
            "COP_thermal": Q_cool/Q_gen, "circulation ratio": fcirc,
            "m_refrigerant, kg/s": m_ref, "m_weak, kg/s": m_weak,
            "energy balance, kW": (Q_cool + Q_gen - Q_cond - Q_abs)/1e3}

ch = chiller(Q_cool)
for k, v in ch.items(): print(f"  {k:22s} {v:10.4f}")
print("\n  The energy balance should close: Q_evap + Q_gen = Q_cond + Q_abs.")


## 3. The collector field

Hottel–Whillier–Bliss: a collector's efficiency falls linearly (near enough)
with the ratio of temperature lift to irradiance. Evacuated tubes have a much
smaller loss coefficient than flat plates, which is why they can reach the
88 °C the generator needs.


In [ ]:
eta_0, a1_, a2_ = 0.72, 1.60, 0.0080     # evacuated tube, typical certified values
T_amb_day = am.K(32)                     # Agra, mean daytime in the storage season

def collector_eff(G, T_m, T_amb=T_amb_day):
    """Hottel-Whillier-Bliss, second order. G in W/m2."""
    if G <= 0: return 0.0
    dT = T_m - T_amb
    return max(eta_0 - a1_*dT/G - a2_*dT**2/G, 0.0)

T_mean_coll = T_gen + 7.0        # collector mean fluid temperature
print(f"{'irradiance G':>14}{'efficiency':>12}{'useful W/m2':>14}")
for G in (300, 500, 700, 900):
    e = collector_eff(G, T_mean_coll)
    print(f"{G:14.0f}{e:12.4f}{e*G:14.1f}")
print(f"\n  A flat plate (eta_0 0.78, a1 4.0) at G = 700 would manage "
      f"{max(0.78 - 4.0*(T_mean_coll-T_amb_day)/700, 0):.3f}")
print("  which is why this machine needs evacuated tubes, not flat plates.")


## 4. Area and storage for an 80% solar fraction

In [ ]:
H_day   = 5.5e3*3600/1e3       # Wh/m2/day -> J/m2/day at Agra (5.5 kWh/m2/day)
hours_sun = 8.0
G_mean  = 5.5e3/hours_sun      # W/m2 averaged over the sunlit period
SF      = 0.80                 # target solar fraction

Q_gen_W  = ch["Q_gen, kW"]*1e3
E_gen_day = Q_gen_W*24*3600                       # J/day the generator needs
E_solar_needed = SF*E_gen_day

eta_coll = collector_eff(G_mean, T_mean_coll)
E_per_m2_day = eta_coll*G_mean*hours_sun*3600     # J/m2/day
A_coll = E_solar_needed/E_per_m2_day

# storage carries the machine through the night
E_night = Q_gen_W*(24 - hours_sun)*3600
dT_store = 25.0                                   # usable swing in the tank
V_store = E_night/(1000.0*4180.0*dT_store)

print(f"  generator duty          {Q_gen_W/1e3:8.2f} kW  (continuous)")
print(f"  daily generator energy  {E_gen_day/3.6e9:8.2f} MWh/day")
print(f"  collector efficiency    {eta_coll:8.4f} at G = {G_mean:.0f} W/m2")
print(f"  useful yield            {E_per_m2_day/3.6e6:8.3f} kWh/m2/day")
print(f"\n  collector area for {SF*100:.0f}% solar fraction: {A_coll:8.1f} m2")
print(f"  storage volume for the night:            {V_store:8.2f} m3")
print(f"  (at a {dT_store:.0f} K usable swing)")


In [ ]:
rows = []
for sf in np.arange(0.3, 1.01, 0.05):
    A = sf*E_gen_day/E_per_m2_day
    rows.append({"solar fraction": float(sf), "collector area, m2": A,
                 "area per TR, m2/TR": A/(Q_cool/3516.85),
                 "storage, m3": V_store,
                 "backup heat needed, kW": (1-sf)*Q_gen_W/1e3})
fig, ax = plt.subplots()
ax.plot([r_["solar fraction"]*100 for r_ in rows],
        [r_["collector area, m2"] for r_ in rows], "o-", lw=2.4, color=am.ORANGE)
ax.axvline(80, color=am.MUTED, ls="--")
ax.plot(80, A_coll, "o", ms=11, color=am.NAVY, zorder=5)
ax.annotate(f"  {A_coll:.0f} m²", (80, A_coll), color=am.NAVY, fontsize=11)
ax.set_xlabel("solar fraction  (%)"); ax.set_ylabel("collector area  (m²)")
ax.set_title("Area is linear in solar fraction; the last 20% costs as much as the first")
plt.tight_layout(); plt.show()
print("  Area scales linearly with solar fraction, but COST does not: the last")
print("  fraction runs only on the worst days, so its capacity factor is awful.")
print("  100% solar is almost never the economic answer. A backup burner is.")


## 5. What this model can and cannot tell you

**It can** show you the effect of the concentration gap, because the circulation
ratio follows from a pure mass balance.

**It cannot** show you the effect of condenser temperature. Raising the
condenser temperature forces a higher generator temperature and *narrows* the
usable concentration gap — but that link runs through the **Dühring
equilibrium**, which is exactly the piece CoolProp does not supply and which you
entered by hand. Hold the concentrations fixed, as here, and the model reports
almost no penalty. That is an artefact of the model, not physics.

Knowing where your model stops being trustworthy is worth more than another
decimal place.


In [ ]:
# The lever the model DOES capture: the concentration gap.
print(f"{'X_strong':>10}{'gap':>8}{'f':>9}{'COP_th':>9}{'Q_gen kW':>11}{'area m2':>10}")
sens = []
for Xs in (0.58, 0.59, 0.60, 0.62, 0.64, 0.66):
    c = chiller(Q_cool, X_s=Xs)
    A = SF*c["Q_gen, kW"]*1e3*24*3600/E_per_m2_day
    sens.append({"X_strong": Xs, "gap": Xs - X_weak, **c, "collector area, m2": A})
    print(f"{Xs:10.3f}{Xs-X_weak:8.3f}{c['circulation ratio']:9.2f}"
          f"{c['COP_thermal']:9.4f}{c['Q_gen, kW']:11.2f}{A:10.1f}")
print("\n  Narrow the gap and the circulation ratio explodes: at a 0.02 gap you")
print("  pump 29 kg of solution per kg of vapour. That is pump power, pipe size")
print("  and solution heat exchanger duty, all rising together.")
print("\n  Now the honest part: run the same sweep on T_cond and you will see")
print("  almost NO effect, because this model holds the concentrations fixed.")
print("  The real condenser penalty travels through the Duhring equilibrium.")
for Tc in (34, 38, 42):
    c = chiller(Q_cool, T_cond=am.K(Tc), T_abs=am.K(Tc-2))
    print(f"    T_cond {Tc} C -> COP_th {c['COP_thermal']:.4f}   <- barely moves. "
          "That is the model's limit, not the machine's.")


## 6. The workbook

In [ ]:
path = am.to_excel("AM5061_D12_AbsorptionChiller.xlsx",
    {"Solar fraction study": rows, "Condenser sensitivity": sens},
    title="AM5061 D-12 . Solar LiBr-water absorption chiller, Agra",
    summary=[("Store capacity", MASS_T, "t"), ("Cooling load", Q_cool/1e3, "kW"),
             ("Cooling load", Q_cool/3516.85, "TR"),
             ("Evaporator / condenser", f"{am.C(T_evap):.0f} / {am.C(T_cond):.0f}", "C"),
             ("Generator / absorber", f"{am.C(T_gen):.0f} / {am.C(T_abs):.0f}", "C"),
             ("Weak / strong concentration", f"{X_weak} / {X_strong}", "kg/kg"),
             ("Circulation ratio", ch["circulation ratio"], "-"),
             ("COP thermal", ch["COP_thermal"], "-"),
             ("Generator duty", ch["Q_gen, kW"], "kW"),
             ("Collector area at 80% SF", A_coll, "m2"),
             ("Storage volume", V_store, "m3")],
    sources=[("Water properties", "CoolProp, IAPWS-95"),
             ("LiBr solution enthalpy", "CoolProp INCOMP::LiBr"),
             ("Concentrations", "READ FROM THE DUHRING CHART - design inputs, not computed"),
             ("Collector", "Hottel-Whillier-Bliss, evacuated tube eta0 0.72, a1 1.60, a2 0.0080"),
             ("Agra irradiance", "5.5 kWh/m2/day, 8 sunlit hours - TYPICAL, use TMY for a real design"),
             ("Potato respiration", "25 W/t at 4 C - typical")])
print("written:", path)


## What to hand in

1. The cycle: circulation ratio, COP_thermal, and all four duties with the
   energy balance closing.
2. Collector area and storage volume for 80% solar fraction.
3. The condenser-temperature sensitivity, and what it implies for the cooling
   tower you would specify.
4. **The concentrations you read off the Dühring chart**, and the crystallisation
   margin at your strong-solution point.
5. The workbook.

**One paragraph:** COP_thermal is about 0.7 while a vapour-compression chiller
manages 3 or more. Explain to a non-engineer why anyone would build this
machine anyway.
